# Projeto MPI — Mandelbrot (exemplo inicial: divisão estática por blocos)

**Disciplina:** DCE540 — Computação Paralela e Distribuída
**Curso:** Bacharelado em Ciência da Computação — UNIFAL-MG

Este notebook é o **ponto de partida** para o projeto de geração paralela do conjunto de Mandelbrot em MPI (grupos de até 3 alunos, 10 dias). Ele contém:

1. Uma versão **sequencial** de referência em C (sem MPI), para comparação de tempo e para conferir a corretude do resultado.
2. Uma versão **paralela**, com divisão **estática por blocos contíguos de linhas** entre os processos — a estratégia mais simples de implementar.
3. Uma medição de **speedup** e uma discussão sobre o **desbalanceamento de carga** que essa estratégia simples apresenta.
4. Sugestões concretas de **evolução** para os grupos continuarem o projeto a partir daqui.

> A versão paralela aqui **funciona e foi testada**, mas contém simplificações propositais (destacadas nos comentários do código). Ela não é a entrega final do projeto — é a base sobre a qual cada grupo deve construir sua própria solução.

## 0. Preparando o ambiente

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libopenmpi-dev openmpi-bin
print("Instalação concluída.")

## 1. O algoritmo e a versão sequencial de referência

O conjunto de Mandelbrot é calculado pixel a pixel pelo **algoritmo do tempo de escape**: cada pixel (px, py) é mapeado para um número complexo c = (cx, cy); a partir de z = 0, itera-se z = z² + c contando as iterações até que |z| ultrapasse 2 (a sequência diverge) ou até um limite `max_iter`. O número de iterações decide a cor do pixel — quem nunca escapa é considerado dentro do conjunto.

Esse cálculo é **independente para cada pixel** (nenhum pixel depende do resultado de outro), o que torna a paralelização direta: basta dividir os pixels entre os processos. A dificuldade está em fazer essa divisão de forma equilibrada, já que pixels diferentes custam quantidades muito diferentes de iterações.

A versão abaixo é sequencial (sem MPI) e serve como **referência de corretude e de tempo**: qualquer versão paralela correta deve produzir exatamente a mesma imagem.

In [ ]:
%%writefile mandelbrot_sequencial.c
#include <stdio.h>

/* Algoritmo do "tempo de escape" para o conjunto de Mandelbrot. */

const int W = 1600, H = 1200;
const double XMIN = -2.5, XMAX = 1.0;
const double YMIN = -1.25, YMAX = 1.25;
const int MAX_ITER = 300;

int escape_time(double cx, double cy) {
    double zx = 0.0, zy = 0.0;
    int iter = 0;

    while (zx * zx + zy * zy <= 4.0 && iter < MAX_ITER) {
        double zx_novo = zx * zx - zy * zy + cx;
        double zy_novo = 2.0 * zx * zy + cy;
        zx = zx_novo;
        zy = zy_novo;
        iter++;
    }
    return iter;
}

/* Converte o numero de iteracoes em uma cor (R,G,B); quem atinge
   MAX_ITER (dentro do conjunto) fica preto. */
void colorir(int iter, unsigned char *r, unsigned char *g, unsigned char *b) {
    if (iter == MAX_ITER) {
        *r = *g = *b = 0;
    } else {
        double t = (double) iter / MAX_ITER;
        *r = (unsigned char) (9 * (1 - t) * t * t * t * 255);
        *g = (unsigned char) (15 * (1 - t) * (1 - t) * t * t * 255);
        *b = (unsigned char) (8.5 * (1 - t) * (1 - t) * (1 - t) * t * 255);
    }
}

int main(void) {
    FILE *arq = fopen("mandelbrot_c.ppm", "wb");
    fprintf(arq, "P6\n%d %d\n255\n", W, H);

    for (int py = 0; py < H; py++) {
        double cy = YMIN + (YMAX - YMIN) * py / (H - 1);
        for (int px = 0; px < W; px++) {
            double cx = XMIN + (XMAX - XMIN) * px / (W - 1);

            int iter = escape_time(cx, cy);

            unsigned char r, g, b;
            colorir(iter, &r, &g, &b);
            fwrite(&r, 1, 1, arq);
            fwrite(&g, 1, 1, arq);
            fwrite(&b, 1, 1, arq);
        }
    }

    fclose(arq);
    printf("Imagem gerada: mandelbrot_c.ppm (%dx%d, max_iter=%d)\n", W, H, MAX_ITER);
    return 0;
}

In [ ]:
!gcc -O2 -o mandelbrot_sequencial mandelbrot_sequencial.c -lm
!time ./mandelbrot_sequencial

O arquivo gerado está no formato **PPM** (P6): um cabeçalho de texto simples seguido dos bytes R, G, B de cada pixel — não precisa de nenhuma biblioteca externa para escrever, e o Python consegue abrir com `PIL.Image.open` sem conversão. Vamos exibir o resultado:

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("mandelbrot_c.ppm")
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Mandelbrot — versão sequencial (referência)")
plt.show()

## 2. Versão paralela — divisão estática por blocos de linhas

A estratégia mais simples de paralelizar: o processo de rank `i` calcula um intervalo **fixo e contíguo** de linhas da imagem (as primeiras `H/size` linhas vão para o rank 0, as próximas `H/size` para o rank 1, e assim por diante). Cada processo calcula seu bloco localmente, sem nenhuma comunicação durante o cálculo, e ao final `MPI_Gather` concatena os blocos — na ordem dos ranks — no processo 0, que grava a imagem.

**Simplificação proposital:** o código exige que `H` seja divisível pelo número de processos, para que o `MPI_Gather` (que exige a mesma contagem de todo mundo) baste. Tratar o caso geral (com blocos de tamanhos diferentes) exigiria `MPI_Gatherv` — fica como uma das sugestões de evolução na Seção 5.

In [ ]:
%%writefile mandelbrot_mpi_blocos.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

/* Versao paralela do algoritmo de Mandelbrot com DIVISAO ESTATICA POR
   BLOCOS DE LINHAS: o processo de rank i calcula um intervalo fixo e
   contiguo de linhas da imagem. Ao final, MPI_Gather concatena os
   blocos (na ordem dos ranks) e o processo 0 grava a imagem em disco.

   SIMPLIFICACAO PROPOSITAL: exige que H seja divisivel pelo numero de
   processos, para que MPI_Gather (com contagem igual para todos) baste.
   Ver as sugestoes de evolucao ao final do notebook para tratar o caso
   geral com MPI_Gatherv. */

const int W = 1600, H = 1200;
const double XMIN = -2.5, XMAX = 1.0;
const double YMIN = -1.25, YMAX = 1.25;
const int MAX_ITER = 300;

int escape_time(double cx, double cy) {
    double zx = 0.0, zy = 0.0;
    int iter = 0;
    while (zx * zx + zy * zy <= 4.0 && iter < MAX_ITER) {
        double zx_novo = zx * zx - zy * zy + cx;
        double zy_novo = 2.0 * zx * zy + cy;
        zx = zx_novo;
        zy = zy_novo;
        iter++;
    }
    return iter;
}

void colorir(int iter, unsigned char *r, unsigned char *g, unsigned char *b) {
    if (iter == MAX_ITER) {
        *r = *g = *b = 0;
    } else {
        double t = (double) iter / MAX_ITER;
        *r = (unsigned char) (9 * (1 - t) * t * t * t * 255);
        *g = (unsigned char) (15 * (1 - t) * (1 - t) * t * t * 255);
        *b = (unsigned char) (8.5 * (1 - t) * (1 - t) * (1 - t) * t * 255);
    }
}

int main(int argc, char **argv) {
    int rank, size;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (H % size != 0) {
        if (rank == 0) {
            fprintf(stderr,
                "Este exemplo exige que H (%d) seja divisivel pelo numero "
                "de processos (%d). Tente -np 1, 2, 4, 5, 8, 10, 16...\n",
                H, size);
        }
        MPI_Finalize();
        return 1;
    }

    int linhas_por_processo = H / size;
    int linha_inicial = rank * linhas_por_processo;

    unsigned char *bloco_local = malloc((size_t) linhas_por_processo * W * 3);

    /* Sincroniza antes de medir, para que todos comecem a cronometrar
       a partir do mesmo instante logico. */
    MPI_Barrier(MPI_COMM_WORLD);
    double t_inicio = MPI_Wtime();

    for (int i = 0; i < linhas_por_processo; i++) {
        int py = linha_inicial + i;
        double cy = YMIN + (YMAX - YMIN) * py / (H - 1);
        for (int px = 0; px < W; px++) {
            double cx = XMIN + (XMAX - XMIN) * px / (W - 1);
            int iter = escape_time(cx, cy);

            unsigned char r, g, b;
            colorir(iter, &r, &g, &b);

            int offset = (i * W + px) * 3;
            bloco_local[offset + 0] = r;
            bloco_local[offset + 1] = g;
            bloco_local[offset + 2] = b;
        }
    }

    double tempo_local = MPI_Wtime() - t_inicio;
    printf("[Processo %d] calculou %d linhas em %.4f s\n",
           rank, linhas_por_processo, tempo_local);
    fflush(stdout);

    unsigned char *imagem_completa = NULL;
    if (rank == 0) {
        imagem_completa = malloc((size_t) H * W * 3);
    }

    /* Cada processo contribui com seu bloco; o Gather concatena na
       ordem dos ranks -- como os blocos sao contiguos e em ordem, o
       resultado remonta a imagem corretamente sem nenhum rearranjo. */
    MPI_Gather(bloco_local, linhas_por_processo * W * 3, MPI_UNSIGNED_CHAR,
               imagem_completa, linhas_por_processo * W * 3, MPI_UNSIGNED_CHAR,
               0, MPI_COMM_WORLD);

    double tempo_max;
    MPI_Reduce(&tempo_local, &tempo_max, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("\n[Processo 0] tempo maximo entre os %d processos: %.4f s\n",
               size, tempo_max);

        FILE *arq = fopen("mandelbrot_mpi.ppm", "wb");
        fprintf(arq, "P6\n%d %d\n255\n", W, H);
        fwrite(imagem_completa, 1, (size_t) H * W * 3, arq);
        fclose(arq);
        printf("[Processo 0] imagem salva em mandelbrot_mpi.ppm\n");

        free(imagem_completa);
    }

    free(bloco_local);
    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -O2 -o mandelbrot_mpi_blocos mandelbrot_mpi_blocos.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./mandelbrot_mpi_blocos

**Repare nos tempos individuais.** Com 4 processos, cada um calcula exatamente 300 linhas — o mesmo número de pixels. Ainda assim, os processos 0 e 3 (que calculam as linhas mais próximas do **topo e da base** da imagem, longe do eixo real) terminaram em ~0,04 s, enquanto os processos 1 e 2 (linhas do **meio** da imagem, onde fica a maior parte do conjunto e da borda fractal densa) levaram ~0,43 s — mais de **10 vezes mais tempo**, calculando a mesma quantidade de pixels. Isso é o desbalanceamento de carga discutido nesse tipo de projeto: dividir o trabalho em partes de mesmo *tamanho* não significa dividir em partes de mesmo *custo*.

### Verificação de corretude

Uma versão paralela correta deve produzir **exatamente** a mesma imagem que a versão sequencial. Como os dois programas escrevem arquivos PPM não comprimidos, basta comparar os arquivos byte a byte:

In [ ]:
!cmp mandelbrot_c.ppm mandelbrot_mpi.ppm && echo "IDÊNTICOS (correção verificada)"

## 3. Medindo o speedup

A célula abaixo roda o programa paralelo várias vezes, variando o número de processos, captura o "tempo máximo" impresso por cada execução e calcula o speedup em relação a `-np 1` (T(1) / T(p)).

> **Sobre os números que você vai ver:** o Colab normalmente oferece só 2 vCPUs. Com `--oversubscribe` conseguimos *lançar* mais processos MPI do que núcleos físicos, mas eles disputam a mesma CPU — por isso o speedup não cresce linearmente com o número de processos, e os tempos variam de uma execução para outra (vale rodar mais de uma vez e observar a variação). Isso não é uma falha do código: é o esperado ao simular paralelismo real num ambiente com poucos núcleos, e reforça por que medir com cuidado (`MPI_Barrier` + várias repetições) importa.

In [ ]:
import re
import subprocess
import matplotlib.pyplot as plt

processos = [1, 2, 4, 8]
tempos = []

for np in processos:
    resultado = subprocess.run(
        ["mpirun", "--allow-run-as-root", "--oversubscribe",
         "-np", str(np), "./mandelbrot_mpi_blocos"],
        capture_output=True, text=True, check=True,
    )
    m = re.search(r"tempo maximo entre os \d+ processos: ([\d.]+) s", resultado.stdout)
    tempo = float(m.group(1))
    tempos.append(tempo)
    print(f"np={np}: {tempo:.4f} s")

t1 = tempos[0]
speedup = [t1 / t for t in tempos]

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(processos, processos, linestyle="--", linewidth=1.5,
        color="#9CA3AF", label="Speedup ideal (linear)")
ax.plot(processos, speedup, marker="o", markersize=8, linewidth=2,
        color="#3B82F6", label="Speedup obtido")
ax.set_xlabel("Número de processos")
ax.set_ylabel("Speedup  (T(1) / T(p))")
ax.set_title("Speedup — Mandelbrot, divisão estática por blocos")
ax.set_xticks(processos)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 4. Por que o desbalanceamento acontece aqui

Na região do plano complexo usada (real de -2.5 a 1.0, imaginário de -1.25 a 1.25), o conjunto de Mandelbrot e sua borda fractal densa ficam concentrados perto do **eixo real** (y ≈ 0) — ou seja, nas linhas do **meio** da imagem. Pontos bem acima ou abaixo (y longe de 0) costumam divergir em poucas iterações. Como a divisão por blocos contíguos dá as linhas do meio inteiras para um ou dois processos, esses processos ficam sistematicamente sobrecarregados, enquanto os que pegam as bordas terminam cedo e ficam ociosos esperando no `MPI_Gather`.

Duas formas (entre outras) de atacar esse problema, cada uma com uma exigência diferente sobre os conceitos já vistos no curso:

- **Divisão intercalada:** em vez de dar as linhas `[0..299]` para o rank 0 e `[300..599]` para o rank 1, distribuir de forma round-robin — a linha `k` vai para o processo `k % size`. Isso mistura linhas caras e baratas entre todos os processos sem precisar de nenhuma comunicação extra durante o cálculo (só muda a conta de qual linha pertence a quem, e como remontar a imagem depois do Gather).
- **Mestre-trabalhadores dinâmico:** o processo 0 distribui linhas (ou pequenos blocos) sob demanda, e cada trabalhador pede a próxima linha assim que termina a anterior. Como o mestre não sabe quem vai terminar primeiro, ele recebe com `MPI_ANY_SOURCE`/`MPI_ANY_TAG` — o padrão da Seção 4 do notebook *MPI_Send_Receive_Formas_e_Tempo*. Essa versão equilibra a carga automaticamente, ao custo de mais comunicação e mais complexidade de código.

## 5. Sugestões de evolução para o projeto

Este notebook **não é a entrega final** — é a base para o grupo evoluir. Algumas direções concretas, da mais simples à mais trabalhosa:

- **Tamanhos não divisíveis:** remover a restrição `H % size == 0` usando `MPI_Gatherv` (contagens e deslocamentos por processo), permitindo qualquer número de processos e qualquer resolução.
- **Divisão intercalada (round-robin):** implementar a alternativa descrita na Seção 4 e comparar o novo perfil de tempos por processo com o da versão por blocos.
- **Mestre-trabalhadores dinâmico:** implementar a versão com `MPI_ANY_SOURCE`/`MPI_ANY_TAG` da Seção 4 e comparar o speedup e o equilíbrio de carga com as duas versões estáticas.
- **Comunicação não bloqueante:** sobrepor o cálculo de um bloco com o envio do bloco anterior usando `MPI_Isend`, em vez de esperar todo mundo terminar para só então fazer o `Gather`.
- **Parametrizar em vez de fixar:** ler resolução, região do plano e `max_iter` por linha de comando (`argv`) em vez de constantes fixas no código, facilitando gerar zooms diferentes sem recompilar.
- **Analisar mais fatores:** repetir a medição de speedup para `max_iter` maior/menor, ou para uma região bem mais "cara" (por exemplo, um zoom só na borda densa), e discutir como isso muda o desbalanceamento observado.